# Electrical Resistivity Tomography (ERT)

**Near-Surface Geophysics — Missouri S&T**

A resistivity survey pushes current into the ground through two electrodes and
measures the voltage it makes between two others. Repeat that for hundreds of
four-electrode combinations along a line of electrodes and you have data that
see to different depths at different places. **ERT** turns those numbers into
a 2-D picture of the ground's resistivity.

This notebook uses a real AGI SuperSting line — 42 electrodes at 3 ft
(0.9144 m), dipole-dipole — exported in the RES2DINV `.dat` format, and the
`shallowgeo.ert` module, which needs only NumPy and SciPy.

1. **Read** the file, and repair the two things wrong with every SuperSting
   export of this kind: non-integer *n*, and readings made with electrodes
   that were never in the ground.
2. **The forward problem**: geometric factors, apparent resistivity, and
   what a pseudosection does and does not show.
3. **Invert** synthetic data recorded with this line's exact measurements,
   to see what the survey can resolve, how to choose the smoothing, and how
   deep it sees.
4. **Check the field data before believing them** — and find out that these
   cannot be a response of the ground.
5. **Invert the field data anyway**, and read the misfit.

Units are **metres** and **ohm-metres** (Ω·m) throughout.

> **The one thing to take away.** An inversion always returns a picture. Whether
> the picture means anything is decided by the misfit and by checks on the data
> that need no inversion at all. Section 4 is the most important section here.

## 0. Setup

Run this cell first. It installs `shallowgeo` if it is missing (or out of
date) and imports what the notebook uses.

In [ ]:
# Checks for the *ERT* code specifically: an older install imports fine and
# then lacks it, and since the version string does not change between builds,
# a plain `pip install` reports "already satisfied".
import importlib
import sys

PACKAGE = ("shallow-geophysics @ "
           "git+https://github.com/Maurer-GEMLab/shallow-geophysics.git")

def shallowgeo_ready():
    for name in [n for n in sys.modules
                 if n == "shallowgeo" or n.startswith("shallowgeo.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    try:
        from shallowgeo.ert import resistivity_tomography  # noqa: F401
        from shallowgeo.datasets import example_data  # noqa: F401
        return True
    except ImportError:
        return False

if not shallowgeo_ready():
    %pip install -q --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    %pip install -q --force-reinstall --no-deps --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    raise ImportError(
        "shallowgeo.ert is still missing after installing. "
        "In Colab: Runtime > Restart session, then run this cell again."
    )

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from shallowgeo.ert import (
    read_res2dinv, dipole_dipole, make_mesh, forward, apparent_resistivity,
    resistivity_tomography, lambda_sweep, doi_index,
)
from shallowgeo.datasets import example_data

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
print("ready")

## 1. The data

By default this uses the **example line that ships with the repository**,
`examples/data/2610234-ert-line/raw/2610234.dat`, exactly as the SuperSting
exported it. Its `dataset.md` lists what is known about the survey and every
known problem.

To use your own line, set `DATA` to the path of a RES2DINV `.dat` file and
`N_ELECTRODES` to the number of electrodes that were actually in the ground.

In [ ]:
DATA = example_data("2610234-ert-line") / "2610234.dat"
N_ELECTRODES = 42

print(*Path(DATA).read_text().splitlines()[:16], sep="\n")

The six header lines are the title, the smallest electrode spacing (0.9144 m
= 3 ft), the array type (`3` = dipole-dipole), the number of measurements, the
x-location convention (`1` = the array's mid-point) and an IP flag. Each data
line is **x, a, n, ρa**: where the array is, its dipole length, the separation
between the dipoles in multiples of *a*, and the apparent resistivity.

Dipole-dipole puts the current pair (C2 C1) on one side and the potential pair
(P1 P2) on the other:

```
  C2 ─a─ C1 ─────── n·a ─────── P1 ─a─ P2
```

Scroll down the raw file and you will find `n` written as `8.00039` or
`2.99999`, and `a` as `6.40079`. The SuperSting computes these from electrode
positions held in single precision. **RES2DINV refuses a non-integer `n`.**

`read_res2dinv` does not trust the `x a n` columns. It works out the four
electrode positions of every measurement, snaps each to the nearest multiple
of the spacing, and numbers the electrodes. That repairs `a`, `n` and `x`
together, and reports how far it had to move anything.

In [ ]:
raw = read_res2dinv(DATA)
print(raw)
print(f"largest snap: {1000 * raw.metadata['snapped_max_m']:.1f} mm")
raw.data.head()

Each row now has the electrode numbers `a b m n` (current pair, potential pair,
counting from 0), the apparent resistivity `rhoa`, the geometric factor `k`,
the transfer resistance `r` = ρa / k = V/I, and the position of the
measurement on a pseudosection (`x`, `depth`).

### Readings past the end of the cable

The file says 136 electrodes. The cable had 42. The SuperSting command file
was written for a longer line, and when an address has no electrode on it the
instrument records a reading anyway. Plot which electrodes each measurement
uses:

In [ ]:
idx = raw.data[list("abmn")].to_numpy()
past = (idx >= N_ELECTRODES).any(axis=1)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.scatter(raw.data["x"][~past], raw.data["depth"][~past], s=12, label="on the cable")
ax.scatter(raw.data["x"][past], raw.data["depth"][past], s=12, color="#d1495b",
           label="uses an electrode that is not there")
ax.axvline(raw.electrodes[N_ELECTRODES - 1, 0], color="k", ls="--", lw=1)
ax.text(raw.electrodes[N_ELECTRODES - 1, 0], 0.3, f" electrode {N_ELECTRODES}", va="top")
ax.set_ylim(raw.data["depth"].max() * 1.05, 0)
ax.set_xlabel("measurement mid-point (m)")
ax.set_ylabel("pseudo-depth (m)")
ax.legend(loc="lower left", fontsize=8)
ax.set_title(f"{past.sum()} of {len(raw)} readings use electrodes past number {N_ELECTRODES}",
             fontsize=10)
plt.show()

A reading can have its mid-point well inside the line and still reach past
the end: a long dipole-dipole spans (n + 2)·a. `trim` drops every measurement
that uses any electrode beyond the last one in the ground, and records what it
did in the data's provenance. `to_res2dinv` writes a file RES2DINV will load:
integer `n`, and a header count that matches.

From a terminal the same repair is one command:
`shallowgeo ert-clean 2610234.dat -e 42`.

In [ ]:
data = raw.trim(N_ELECTRODES)
print(data)
print(data.provenance[-1])
data.to_res2dinv("2610234_clean.dat")
print(*Path("2610234_clean.dat").read_text().splitlines()[:9], sep="\n")

## 2. The forward problem

### Apparent resistivity

Current *I* injected at one electrode on uniform ground of resistivity ρ
raises the potential at distance *r* to V = ρI / (2πr). With two current and
two potential electrodes the voltage measured is

$$ V = \frac{\rho I}{2\pi}\left(\frac{1}{AM} - \frac{1}{BM} - \frac{1}{AN} + \frac{1}{BN}\right)
   \quad\Rightarrow\quad \rho_a = k\,\frac{V}{I}, \qquad
   k = \frac{2\pi}{1/AM - 1/BM - 1/AN + 1/BN}. $$

For dipole-dipole, k = π a n(n+1)(n+2): it grows as n³, because the voltage
falls as n⁻³. Over uniform ground ρa is ρ for every measurement. Over real
ground it is a weighted average of everything the current flows through — an
*apparent* resistivity.

### The pseudosection

Plotting each ρa at its mid-point and at its **median depth of investigation**
(Edwards, 1977; about 0.42 a for n = 1, 1.7 a for n = 6) gives a
*pseudosection*. It is a picture of the data, not of the ground.

### Modelling

`forward` solves for the potential on a finite-element mesh. The ground is
2-D but the source is a point, so the potential is 3-D; a Fourier transform
across the line turns it into a set of 2-D problems, one per wavenumber
(the "2.5-D" method). The mesh is a quarter of an electrode spacing wide under
the electrodes and grows geometrically outwards. Its error against the exact
solution for two layers is under 1 %.

Here is what a buried conductor does to a dipole-dipole pseudosection,
recorded with **this line's 214 measurements**:

In [ ]:
mesh = make_mesh(data)
print(mesh)

def model(X, Z):
    """80 ohm-m soil over 400 ohm-m, with a 20 ohm-m body at 20 m."""
    rho = np.where(Z < 1.5, 80.0, 400.0)
    return np.where((np.abs(X - 20) < 2.5) & (Z > 1.5) & (Z < 4), 20.0, rho)

X, Z = mesh.mesh()
true = model(X, Z)
synthetic = apparent_resistivity(data, model, mesh=mesh, noise=0.03, seed=1)

fig, axes = plt.subplots(2, 1, figsize=(10, 6.5))
im = axes[0].pcolormesh(mesh.x_edges, mesh.z_edges, true, cmap="turbo",
                        norm=plt.matplotlib.colors.LogNorm(20, 400))
fig.colorbar(im, ax=axes[0], label="resistivity (Ω·m)", shrink=0.8, pad=0.02)
axes[0].set_ylim(10, 0); axes[0].set_aspect("equal")
axes[0].set_title("true model", fontsize=10)
axes[0].set_ylabel("depth (m)")
synthetic.plot_pseudosection(axes[1])
axes[1].set_title("its pseudosection (3 % noise), this line's measurement scheme", fontsize=10)
plt.tight_layout(); plt.show()

Notice the measurement scheme itself. It is dense under the first 15 m of the
line and sparse beyond — the command file was written for a different line
length, so most of the long dipoles it planned were the ones that fell off
the end of the cable. The body at 20 m sits where coverage is thinning.

## 3. Inversion

The model is the logarithm of resistivity in each cell of the model grid.
Each Gauss-Newton step solves

$$ \min_m\; \left\| \frac{\ln\rho_a^{obs} - \ln\rho_a(m)}{e} \right\|^2
   + \lambda \,\|R\,m\|^2 $$

where *e* is the relative data error (3 % here) and R takes differences
between neighbouring cells, vertical ones weighted by `z_weight`. λ trades
fit against smoothness. The sensitivities come from the same potentials as
the forward solve (by reciprocity), so each step costs about two forward
solves.

In [ ]:
res = resistivity_tomography(synthetic, mesh=mesh, lam=20, verbose=True)
print(res.summary())

fig, axes = plt.subplots(2, 1, figsize=(10, 6.5))
res.plot(axes[0], vmin=20, vmax=400, max_depth=10)
axes[0].set_title("recovered from the synthetic data", fontsize=10)
res.plot(axes[1], field="coverage", max_depth=10)
axes[1].set_title("relative coverage (error-weighted cumulative sensitivity)", fontsize=10)
plt.tight_layout(); plt.show()

The image is **faded where coverage falls below 1 % of its maximum** — those
colours are the smoothness constraint filling in, not the ground. The layer
boundary comes back as a gradient, the body comes back larger and less
extreme than it is, and the body's base is not resolved. That is what a
smoothness-constrained inversion does; it is not a bug.

### Choosing λ

Too small and the image grows artefacts to fit the noise; too large and it
cannot fit the data. The usual choice is the smoothest model that fits the
data to the noise level, chi² ≈ 1.

In [ ]:
sweep = lambda_sweep(synthetic, lams=(2, 6, 20, 60, 200), mesh=mesh)
print(sweep[["lam", "chi2", "rms_pct", "roughness"]].round(2))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
axes[0].loglog(sweep["roughness"], sweep["chi2"], "o-")
for _, r in sweep.iterrows():
    axes[0].annotate(f"λ={r['lam']:g}", (r["roughness"], r["chi2"]), fontsize=8)
axes[0].axhline(1, color="k", ls="--", lw=0.8)
axes[0].set_xlabel("model roughness |Rm|"); axes[0].set_ylabel("chi²")
for ax, i in zip(axes[1:], (0, len(sweep) - 1)):
    sweep["result"].iloc[i].plot(ax, vmin=20, vmax=400, max_depth=8, colorbar=False)
    ax.set_title(f"λ = {sweep['lam'].iloc[i]:g}", fontsize=10)
plt.tight_layout(); plt.show()

### How deep does it see?

Coverage is one guide. A stricter one is the **depth-of-investigation index**
(Oldenburg & Li, 1999): invert twice, starting from and pulled weakly towards
reference models ten times above and ten times below the median apparent
resistivity. Where the data control the model both runs agree (index near 0);
where they do not, each stays near its own reference (index near 1).

In [ ]:
doi, low, high = doi_index(synthetic, mesh=mesh, lam=20)

fig, axes = plt.subplots(2, 1, figsize=(10, 6.5))
im = axes[0].pcolormesh(mesh.x_edges, mesh.z_edges, doi, cmap="viridis", vmin=0, vmax=1)
axes[0].contour(X, Z, doi, levels=[0.2], colors="w")
fig.colorbar(im, ax=axes[0], label="DOI index", shrink=0.8, pad=0.02)
axes[0].set_ylim(mesh.z_edges[-1], 0); axes[0].set_aspect("equal")
axes[0].set_ylabel("depth (m)")
axes[0].set_title("DOI index (white: 0.2)", fontsize=10)
res.plot(axes[1], mask=doi < 0.2, vmin=20, vmax=400)
axes[1].set_title("the image, faded where DOI > 0.2", fontsize=10)
plt.tight_layout(); plt.show()

## 4. Check the field data before inverting them

Nothing so far needed the field data. Before inverting them, look at them in
ways that need no inversion at all. Here are the dipole-length-1 readings,
apparent resistivity and transfer resistance, against n:

In [ ]:
d = data.data
dip = (d["a"] - d["b"]).abs()
sep = (d["m"] - d["a"]) // dip
d1 = d[dip == 1]
by_n = d1.groupby(sep[dip == 1])[["rhoa", "r"]].median()
by_n.index.name = "n"
print(by_n.round(2))

# The most extreme layered ground there is: a thin conductive layer on a
# perfect insulator. How fast can dipole-dipole rho_a rise over it?
dd = dipole_dipole(N_ELECTRODES, data.spacing, n_max=8)
mesh_dd = make_mesh(dd, depth=15)
_, Zdd = mesh_dd.mesh()
dd_sep = (dd.data["m"] - dd.data["a"]).to_numpy()
extreme = {}
for h in (0.25, 1.0, 2.0):
    rhoa = forward(mesh_dd, np.where(Zdd < h, by_n["rhoa"].iloc[0], 1e6), dd)["rhoa"]
    extreme[h] = [np.median(rhoa[dd_sep == n]) for n in by_n.index]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(by_n.index, by_n["rhoa"], "o-", color="#d1495b", label="field data")
for h, v in extreme.items():
    axes[0].semilogy(by_n.index, v, "--", label=f"{h:g} m of soil on an insulator")
axes[0].axhline(by_n["rhoa"].iloc[0], color="0.5", lw=0.8, label="uniform ground")
axes[0].set_xlabel("n"); axes[0].set_ylabel("apparent resistivity (Ω·m)")
axes[0].legend(fontsize=8)
n = by_n.index.to_numpy()
k = np.pi * data.spacing * n * (n + 1) * (n + 2)
axes[1].semilogy(n, by_n["r"], "o-", color="#d1495b", label="field data")
axes[1].semilogy(n, by_n["rhoa"].iloc[0] / k, color="0.5", label="uniform ground")
axes[1].set_xlabel("n"); axes[1].set_ylabel("transfer resistance V/I (Ω)")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

The field apparent resistivity rises **35-fold** from n = 1 to n = 8, the
same at every position along the line. Even a thin conductive layer on a
perfect insulator — the most extreme layered ground there is — can only raise
dipole-dipole apparent resistivity three- to fivefold over the same range (and
2-D structure cannot do it everywhere along the line at once).

The right-hand plot says why. The voltage per unit current should fall about
a hundredfold between n = 1 and n = 8, because the potential electrodes move
away from the current. It barely falls at all.

One more test, and it is decisive. Compare the readings made with electrodes
that were in the ground to the 109 that `trim` removed, whose potential
electrodes were attached to nothing:

In [ ]:
inside = ~past
cur_on_cable = (idx[:, :2] < N_ELECTRODES).all(axis=1)
print(f"removed readings with both current electrodes on the cable: "
      f"{(past & cur_on_cable).sum()} of {past.sum()}")

fig, ax = plt.subplots(figsize=(7, 3.5))
bins = np.geomspace(1, 7, 30)
ax.hist(raw.data["r"][inside], bins=bins, alpha=0.6, label=f"electrodes in the ground ({inside.sum()})")
ax.hist(raw.data["r"][past], bins=bins, alpha=0.6, color="#d1495b",
        label=f"electrodes not there ({past.sum()})")
ax.set_xscale("log"); ax.set_xlabel("transfer resistance V/I (Ω)"); ax.set_ylabel("readings")
ax.legend(fontsize=8)
plt.show()
print(raw.data.groupby(np.where(past, "not there", "in the ground"))["r"].describe().round(2))

**The two are indistinguishable.** A potential electrode attached to nothing
cannot measure the ground, yet those readings look exactly like the rest. So
the recorded voltages are dominated by something other than the ground under
the potential electrodes — leakage between conductors in a wet or damaged
cable or connector, a mismatch between the cable and the addresses the
command file uses, or the wrong cable type set in the instrument. Trimming
removes the readings we *know* are not data; it cannot make the rest real.

And the last check. The SuperSting measures up to eight potential dipoles at
once for each current injection, one per receiver channel. Tabulate the
median transfer resistance by dipole length and n:

In [ ]:
d = data.data.copy()
d["dipole"] = (d["a"] - d["b"]).abs()
d["n_sep"] = (d["m"] - d["a"]) // d["dipole"]
table = d.pivot_table(index="n_sep", columns="dipole", values="r", aggfunc="median")
print("median transfer resistance V/I (ohm) by n (rows) and dipole length (columns):")
print(table.round(2))

Read along the diagonals: (a, n) = (1, 3), (2, 5), (3, 6) and (4, 7) all give
**3.07 Ω**, although their geometric factors differ tenfold, and the same
short series of values (3.07, 2.81, 2.59, 2.37, 2.10) repeats in every
column. The transfer resistance follows the reading's place in the
eight-channel set, not where its electrodes are. Ground cannot do that; an
instrument or cable fault can.

What would settle it: the SuperSting `.stg` file (the current and the repeat
error of every reading), a contact-resistance test on the cable, and a short
line re-measured with a command file written for 42 electrodes.

## 5. Invert the field data anyway

An inversion run on impossible data still returns a picture. The way to tell
is the misfit.

In [ ]:
field = resistivity_tomography(data, mesh=mesh, lam=20, verbose=True)
print(field.summary())

fig = plt.figure(figsize=(10, 12))
gs = fig.add_gridspec(4, 1)
ax0 = fig.add_subplot(gs[0])
field.plot(ax0)
ax0.set_title(f"field data: RMS misfit {field.rms:.0f} %, chi² {field.chi2:.0f} — do not interpret",
              fontsize=10)
field.plot_pseudosections([fig.add_subplot(gs[i]) for i in (1, 2, 3)])
plt.tight_layout(); plt.show()

For comparison, the synthetic inversion in section 3 fit its 214 readings to
an RMS of about 3 %, the noise that was added. Here the misfit stalls near
40 % and chi² near 170 — about sixty times the noise level — and lowering λ
does not help: there is no 2-D resistivity model whose response looks like
these data. (Two independent codes do no better on the same data: pyGIMLi
stops at chi² 164, RMS 36 %, and ResIPy's R2 at chi² 206 or more, RMS 36 %
or more; see `docs/ert-validation.md`.) The measured pseudosection rises
steadily with depth; the best the model can do is a strongly resistive
basement whose response flattens out, and the misfit panel shows the
systematic pattern left over.

## Summary

- **Repairing the file** — integer *n* and dropping readings past the cable —
  is `read_res2dinv(path, n_electrodes=42)` in Python or
  `shallowgeo ert-clean path -e 42` in a terminal. Do it to every SuperSting
  export whose command file was written for a different line.
- **The inversion works** on data the ground could have produced: the
  synthetic test recovers a conductor from this line's own measurement
  scheme, and shows what that scheme cannot see.
- **This line's data are not a response of the ground.** Three checks that
  need no inversion say so — the growth of ρa with n, the flat transfer
  resistance that follows the receiver channel rather than the geometry, and
  readings from electrodes that were not there looking just like the rest —
  and the inversion agrees by failing to fit. Check the
  instrument records and the cable before collecting the next line.